In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
projdir = '/u/project/cluo/terencew/claude/igvf/pgp/ipsc_le_se_rerun'
outdir = f'{projdir}/txt'
# per-nucleus strand-merged CpG allc (same files the published pseudobulks merged); the originals in
# 2023_YR2/snmCT/mc/allc/raw were removed 2026-10-14, the rerun read them from the weekly NFS snapshot
allcdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/.snapshot/weekly_2026-10-11_12_00_00_UTC/allc/raw'
contactdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/hicluster/rmbkl'
donors = ['C29', 'C37', 'C38', 'C39']
subtypes = ['IPS_main', 'IPS_alt']

### early-iPSC nuclei from R01a

In [ ]:
early = pd.read_csv(f'{projdir}/csv/ipsc_le_se_early_ipsc_nuclei.csv', sep='\t', index_col=0)
early.groupby(['assay', 'ips_subtype']).size()

### snmCT methylation pseudobulks: per subtype and per subtype x donor (CGN-Merge allc)

In [ ]:
mct = early[early['assay'] == 'snmCT'].copy()
mct['allc'] = [f'{allcdir}/{x}.CGN-Merge.allc.tsv.gz' for x in mct.index]
mct['exists'] = [os.path.exists(x) for x in mct['allc']]
mct.groupby(['ips_subtype', 'line'])['exists'].agg(['sum', 'size'])

In [ ]:
os.makedirs(f'{outdir}/allc_table_cgn/ips_cluster', exist_ok=True)
os.makedirs(f'{outdir}/allc_table_cgn/ips_cluster_donor', exist_ok=True)
mct = mct[mct['exists']].sort_index()
for s in subtypes:
    np.savetxt(f'{outdir}/allc_table_cgn/ips_cluster/{s}.txt', mct[mct['ips_subtype'] == s]['allc'].values, fmt='%s')
    for d in donors:
        x = mct[(mct['ips_subtype'] == s) & (mct['line'] == d)]
        np.savetxt(f'{outdir}/allc_table_cgn/ips_cluster_donor/{s}_{d}.txt', x['allc'].values, fmt='%s')
np.savetxt(f'{outdir}/allc_table_cgn/ips_cluster.txt', subtypes, fmt='%s')
np.savetxt(f'{outdir}/allc_table_cgn/ips_cluster_donor.txt', [f'{s}_{d}' for s in subtypes for d in donors], fmt='%s')

### snm3C contact pseudobulks: per subtype

In [ ]:
m3c = early[early['assay'] == 'snm3C'].copy()
m3c['contacts'] = [f'{contactdir}/{x}.contact.rmbkl.tsv.gz' for x in m3c.index]
m3c['exists'] = [os.path.exists(x) for x in m3c['contacts']]
m3c.groupby('ips_subtype')['exists'].agg(['sum', 'size'])

In [ ]:
os.makedirs(f'{outdir}/contacts/ips_cluster', exist_ok=True)
m3c = m3c[m3c['exists']].sort_index()
for s in subtypes:
    np.savetxt(f'{outdir}/contacts/ips_cluster/{s}.txt', m3c[m3c['ips_subtype'] == s]['contacts'].values, fmt='%s')
np.savetxt(f'{outdir}/contacts/ips_cluster.txt', subtypes, fmt='%s')

### list sizes

In [ ]:
{f: sum(1 for _ in open(f'{outdir}/{f}')) for f in ['allc_table_cgn/ips_cluster/IPS_main.txt', 'allc_table_cgn/ips_cluster/IPS_alt.txt', 'contacts/ips_cluster/IPS_main.txt', 'contacts/ips_cluster/IPS_alt.txt']}

In [ ]:
!date